<a href="https://colab.research.google.com/github/Minakshi654/DocSense--CNN-based-Document-Classifier/blob/main/Rag_for_coco_cola.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-community langchain-text-splitters chromadb sentence-transformers google-generativeai -q

import torch
print("GPU available:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/

In [3]:
import os

report_files = {
    "North": {"file": "report_north.txt", "region": "North"},
    "West": {"file": "report_west.txt", "region": "West"},
    "East_South": {"file": "report_east_south.txt", "region": "East_South"},
    "Cross_Regional": {"file": "report_cross_regional_summary.txt", "region": "All"},
}

documents = {}
for name, info in report_files.items():
    with open(info["file"], "r") as f:
        documents[name] = {"text": f.read(), "region": info["region"]}

for name, doc in documents.items():
    print(f"{name}: {len(doc['text'])} characters, region tag = '{doc['region']}'")

North: 3793 characters, region tag = 'North'
West: 2937 characters, region tag = 'West'
East_South: 3074 characters, region tag = 'East_South'
Cross_Regional: 3821 characters, region tag = 'All'


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "],
)

all_chunks = []
all_metadatas = []
chunk_ids = []

chunk_counter = 0
for doc_name, doc_info in documents.items():
    chunks = text_splitter.split_text(doc_info["text"])

    for chunk in chunks:
        all_chunks.append(chunk)
        all_metadatas.append({
            "region": doc_info["region"],
            "source_document": doc_name,
        })
        chunk_ids.append(f"chunk_{chunk_counter}")
        chunk_counter += 1

print("Total chunks across all 4 documents:", len(all_chunks))

# Show the region breakdown
from collections import Counter
region_counts = Counter(m["region"] for m in all_metadatas)
print("\nChunks per region tag:")
for region, count in region_counts.items():
    print(f"  {region}: {count}")

Total chunks across all 4 documents: 42

Chunks per region tag:
  North: 11
  West: 8
  East_South: 9
  All: 14


In [5]:
import chromadb
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="coca_cola_regional_reports")

chunk_embeddings = embedding_model.encode(all_chunks).tolist()

collection.add(
    documents=all_chunks,
    embeddings=chunk_embeddings,
    metadatas=all_metadatas,
    ids=chunk_ids,
)

print("Chunks stored:", collection.count())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunks stored: 42


In [6]:
def search_reports(query, region_filter=None, n_results=3):
    query_embedding = embedding_model.encode([query]).tolist()

    where_clause = {"region": region_filter} if region_filter else None

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where_clause,
    )

    print(f"Query: {query}")
    print(f"Region filter: {region_filter or 'None (searching all regions)'}\n")
    for i, doc in enumerate(results["documents"][0]):
        meta = results["metadatas"][0][i]
        print(f"--- Match {i+1} (region: {meta['region']}, source: {meta['source_document']}) ---")
        print(doc[:200], "...\n")

search_reports("Should we invest in forecasting?")
print("\n" + "="*80 + "\n")
search_reports("Should we invest in forecasting?", region_filter="West")

Query: Should we invest in forecasting?
Region filter: None (searching all regions)

--- Match 1 (region: All, source: Cross_Regional) ---
1. Prioritize advanced demand forecasting investment in West Region and any future identified high-volatility satellite markets, where the LSTM demonstrates clear, substantial value over simpler metho ...

--- Match 2 (region: All, source: Cross_Regional) ---
KEY INSIGHT: The LSTM forecasting model's value is concentrated specifically in smaller, higher-volatility markets, rather than providing uniform benefit across all regions. This supports a segmented  ...

--- Match 3 (region: West, source: West) ---
RECOMMENDATION: The West Region should be prioritized for continued investment in LSTM-based demand forecasting, as it represents the segment where advanced modeling demonstrably outperforms simpler a ...



Query: Should we invest in forecasting?
Region filter: West

--- Match 1 (region: West, source: West) ---
RECOMMENDATION: The West Region shou

In [7]:
from google.colab import userdata
import google.generativeai as genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

def rag_answer(query, region_filter=None, n_results=3):
    query_embedding = embedding_model.encode([query]).tolist()
    where_clause = {"region": region_filter} if region_filter else None

    results = collection.query(query_embeddings=query_embedding, n_results=n_results, where=where_clause)
    retrieved_chunks = results["documents"][0]
    metadatas = results["metadatas"][0]

    context = "\n\n".join(retrieved_chunks)
    region_note = f" (filtered to {region_filter} region only)" if region_filter else " (across all regions)"

    prompt = f"""You are a market intelligence assistant for a CPG company. Answer the question using ONLY the context provided below{region_note}. If the answer isn't in the context, say so.

Context:
{context}

Question: {query}

Answer:"""

    model = genai.GenerativeModel("gemini-2.5-flash")
    response = model.generate_content(prompt)

    print(f"Question: {query}{region_note}\n")
    print(f"Answer: {response.text}\n")
    print("Sources:", [f"{m['source_document']}" for m in metadatas])
    return response.text

rag_answer("Which region should get the most forecasting investment, and why?")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Question: Which region should get the most forecasting investment, and why? (across all regions)

Answer: The West Region should get the most forecasting investment. It is a priority candidate for advanced demand-planning investment, and demand-planning resources should be concentrated on higher-volatility markets like West Region. Additionally, despite its smaller size, the West Region shows the strongest demand forecasting performance among all four tracked regions.

Sources: ['East_South', 'East_South', 'West']


'The West Region should get the most forecasting investment. It is a priority candidate for advanced demand-planning investment, and demand-planning resources should be concentrated on higher-volatility markets like West Region. Additionally, despite its smaller size, the West Region shows the strongest demand forecasting performance among all four tracked regions.'

In [8]:
import pickle

# Save the chunk/metadata structure for reuse in Component 5 (the agent crew)
rag_artifacts = {
    "chunks": all_chunks,
    "metadatas": all_metadatas,
    "documents": documents,
}

with open("rag_artifacts.pkl", "wb") as f:
    pickle.dump(rag_artifacts, f)

from google.colab import files
files.download("rag_artifacts.pkl")

print("RAG artifacts saved for reuse in Component 5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RAG artifacts saved for reuse in Component 5


# Component 4 — Multi-Document RAG with Metadata Filtering

Part of the Coca-Cola Demand Intelligence & Market Insights Platform. A retrieval-augmented generation system spanning 4 regional market intelligence reports, with metadata-based filtering enabling region-scoped or cross-regional question answering.

## Objective

Extend a prior single-document RAG system to support multiple documents with structured metadata, enabling stakeholders to query either a specific region or synthesize insights across all regions — directly addressing a real limitation of single-document RAG when an organization has multiple, regionally-distinct reports.

## Source documents

Four market intelligence reports, written using **real, traceable results** from Components 1–3 of this platform (not synthetic placeholder data):

- **North Region (Flagship)** — incorporating the LSTM's 3.93% MAPE forecast result and the finding that naive baseline forecasting performed comparably
- **West Region (Satellite)** — incorporating the LSTM's 3.36% MAPE result and its substantial outperformance of the 7.41% naive baseline
- **East & South Regions** — incorporating their respective 3.00% and 4.54% MAPE results
- **Cross-Regional Executive Summary** — synthesizing forecasting, compliance (91.17% CNN accuracy), and sentiment (76.8% classifier accuracy) findings across all four regions

42 total chunks generated across all 4 documents (chunk size 500, overlap 50), each tagged with `region` and `source_document` metadata.

## Approach

- Same chunking and embedding pipeline as a prior single-document RAG project (LangChain `RecursiveCharacterTextSplitter`, `all-MiniLM-L6-v2` embeddings, ChromaDB)
- **New: metadata attached to every chunk** (`region`, `source_document`), enabling ChromaDB's `where` clause to restrict retrieval to a specific region before similarity ranking occurs
- Generation via Gemini 2.5 Flash, with the same "answer only from provided context" grounding constraint as the prior RAG project

## Validation — filtered vs. unfiltered retrieval

For the query "Should we invest in forecasting?":
- **Unfiltered search** returned a blend of Cross-Regional summary chunks and one West-specific chunk, reflecting genuinely relevant content across multiple documents
- **Filtered search (`region_filter="West"`)** returned exclusively West Region chunks (3 of 3 matches), with zero leakage from other documents — confirming the metadata filter genuinely constrains the search space rather than merely reordering unfiltered results

## Cross-region synthesis test

A comparison query ("Compare West Region's forecasting performance to North Region's") was tested using unfiltered search, requiring the system to draw on information from multiple regional documents simultaneously. The system correctly retrieved and cited the specific MAPE figures for both regions (West: 3.36% LSTM vs. 7.41% naive; North: 3.93% LSTM vs. 3.41% naive) and correctly identified that West Region benefits substantially more from advanced forecasting than North Region — demonstrating the system can synthesize across document boundaries, not just retrieve within a single document.

## Key finding

Metadata filtering transforms a RAG system from "search everything" into a structurally-aware retrieval system that respects organizational boundaries (in this case, regional reporting structures) — a necessary capability for any real enterprise deployment where a single global search would blend together insights that should sometimes be kept region-specific (e.g., a North Region stakeholder asking about their region shouldn't receive West Region recommendations blended into the answer).

## What I'd improve next

- Add date-based metadata filtering in addition to region, supporting queries like "what changed in the last quarter"
- Add a query-routing step that automatically detects when a question implies a specific region (e.g., "West Region" mentioned in the query) and applies the filter automatically, rather than requiring it to be specified explicitly
- Scale to more documents and test retrieval precision/recall systematically with a labeled evaluation set, rather than spot-checking individual queries

## Tech stack

`Python` · `LangChain` (text splitting) · `ChromaDB` (vector store with metadata filtering) · `Sentence Transformers` · `Google Gemini 2.5 Flash` · Google Colab (T4 GPU)

## How to run

1. Open `04_coca_cola_multidoc_rag.ipynb` in Google Colab
2. Upload the 4 region report `.txt` files
3. Add your Gemini API key as a Colab secret
4. Run all cells in order